# Phase 3: parallel researchers and per-model rate limits

Runs the Phase 3 graph on **the same topic as Phases 1 and 2**, so the three briefs can be compared, and keeps the record of the run:

1. the brief, rendered below and saved to `notebooks/phase_03_brief.md`;
2. the run numbers, now with the time of each stage, when each parallel branch finished, what each branch spent its time on, why pages were skipped, retries by model tier, and the model requests used of the run's budget;
3. the link to the run's trace in LangSmith;
4. a comparison of Phases 1, 2, and 3 on this topic, with this run's numbers;
5. at the end, **the phase record**: the problems met while building Phase 3 and what was decided.

What changed since Phase 2: one `web_researcher` branch per sub-question, all running at the same time (`parallel-fan-out`, D-42); one rate limiter per model, at 80% of its measured limit, and a budget of model requests per run (`shared-throttle`, D-43).

**Before running:** start `search-mcp` in its own terminal and leave it running:

```powershell
uv run python -m mcp_servers.search_mcp.server
```

Needs real keys in `.env`. One run costs up to 14 model requests (at most 30 allowed) and 6 Tavily searches, and takes about 45 to 60 s.

Outputs are stripped from this file when it is committed (`nbstripout`); the brief file is the committed record. Concepts: `parallel-fan-out`, `shared-throttle`.

In [ ]:
# Work from the project root, wherever Jupyter started: settings.py reads .env
# from the working folder, and the graph package is imported from the root.
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT.name}")

In [ ]:
import logging
import re
import socket
import time
from urllib.parse import urlsplit
from uuid import uuid4

from IPython.display import Markdown, display
from langchain_core.tracers.langchain import wait_for_all_tracers

from graph.llm import MODEL_CALL_BUDGET_KEY, ModelGateway
from graph.mcp_client import START_SERVER_COMMAND, McpToolbox
from graph.nodes.researcher import PAGES_PER_QUESTION, SEARCH_RESULTS_PER_QUESTION
from graph.prompts import planner as planner_prompt
from graph.prompts import web_findings as web_findings_prompt
from graph.prompts import writer as writer_prompts
from graph.report import cited_finding_ids, render_brief
from graph.supervisor import build_graph, research_run_config
from settings import settings


class RunLogRecorder(logging.Handler):
    """Keeps the graph's log messages, so skips, drops, retries, and timings can be read."""

    def __init__(self) -> None:
        super().__init__(level=logging.INFO)
        self.messages: list[str] = []

    def emit(self, record: logging.LogRecord) -> None:
        self.messages.append(record.getMessage())


# Every module under graph/ logs to a child of the "graph" logger.
graph_logger = logging.getLogger("graph")
graph_logger.setLevel(logging.INFO)
for old_recorder in [h for h in graph_logger.handlers if isinstance(h, RunLogRecorder)]:
    graph_logger.removeHandler(old_recorder)  # re-running this cell must not count twice
run_log = RunLogRecorder()
graph_logger.addHandler(run_log)

In [ ]:
# CONCEPT: mcp-client. Check that search-mcp is listening before spending any
# model request; McpToolbox.connect would also refuse, with the same command.
server_address = urlsplit(settings.vre_search_mcp_url)
try:
    with socket.create_connection((server_address.hostname, server_address.port or 80), 1.0):
        print(f"search-mcp is running at {settings.vre_search_mcp_url}")
except OSError:
    raise RuntimeError(
        f"search-mcp is not running; start it with: {START_SERVER_COMMAND}"
    ) from None

In [ ]:
TOPIC = "Due diligence on Symbotic: business model, customers, and main risks"

# CONCEPT: tracing-from-day-one. The run id is chosen before the run, so the
# trace can be looked up afterwards. The config also carries this run's fresh
# model-request budget (shared-throttle, D-43), read back after the run.
run_id = uuid4()
run_config = research_run_config(TOPIC, run_id)
run_log.messages.clear()

# CONCEPT: parallel-fan-out. "updates" yields each node's result as soon as it
# finishes, so a branch's time is when its update arrives; "values" yields the
# whole state after each step, and the last one is the final state.
finished_at_by_node: dict[str, float] = {}
branch_finished_at: list[tuple[str, float]] = []
final_state: dict = {}
started_at = time.perf_counter()

# CONCEPT: tools-through-mcp. The notebook owns the toolbox's one connection;
# the graph only uses it (D-25, D-37). The connection closes when the block ends.
async with McpToolbox.connect(settings.vre_search_mcp_url) as toolbox:
    graph = build_graph(ModelGateway(), toolbox)
    async for stream_mode, chunk in graph.astream(
        {"topic": TOPIC, "document_ids": []},
        config=run_config,
        stream_mode=["updates", "values"],
    ):
        if stream_mode == "values":
            final_state = chunk
            continue
        seconds_so_far = time.perf_counter() - started_at
        for node_name, node_update in chunk.items():
            finished_at_by_node[node_name] = seconds_so_far
            if node_name == "web_researcher" and node_update["findings"]:
                sub_question_id = node_update["findings"][0].sub_question_id
                branch_finished_at.append((sub_question_id, seconds_so_far))

elapsed_seconds = time.perf_counter() - started_at
print(f"Done in {elapsed_seconds:.1f} s")

In [ ]:
brief_markdown = render_brief(TOPIC, final_state["draft"], final_state["findings"])
display(Markdown(brief_markdown))

In [ ]:
def count_messages(fragment: str) -> int:
    return sum(fragment in message for message in run_log.messages)


def dropped_by_reason() -> dict[str, int]:
    reasons = re.findall(r"dropped a finding \(([^)]+)\)", "\n".join(run_log.messages))
    return {reason: reasons.count(reason) for reason in sorted(set(reasons))}


SKIP_KINDS = ("timed out", "HTTP", "not a text page", "no readable text", "could not fetch")


def skipped_by_reason() -> dict[str, int]:
    """Why pages were skipped, by kind; the reason text comes from search-mcp."""
    counts: dict[str, int] = {}
    for message in run_log.messages:
        if ": skipped " in message:
            # "Q2: skipped <url> (<reason>)"; a url has no spaces, so the first
            # " (" opens the reason, even one with brackets inside.
            reason = message.split(" (", 1)[1]
            kind = next((known for known in SKIP_KINDS if reason.startswith(known)), "other")
            counts[kind] = counts.get(kind, 0) + 1
    return counts


findings = final_state["findings"]
kept_ids = {finding.id for finding in findings if finding.found}
retry_messages = [message for message in run_log.messages if "transient error" in message]
call_budget = run_config["configurable"][MODEL_CALL_BUDGET_KEY]
planner_seconds = finished_at_by_node["planner"]
research_seconds = finished_at_by_node["research_check"] - planner_seconds
writer_seconds = elapsed_seconds - finished_at_by_node["research_check"]

run_numbers = {
    "sub-questions": len(final_state["sub_questions"]),
    "branches run": len(branch_finished_at),
    "pages skipped": count_messages(": skipped "),
    "searches failed": count_messages(": search failed"),
    "findings kept": len(kept_ids),
    "sub-questions with no evidence": sum(not finding.found for finding in findings),
    "findings dropped by the checks": sum(dropped_by_reason().values()),
    "sources cited": len(cited_finding_ids(final_state["draft"]) & kept_ids),
    "sections": len(final_state["draft"]),
    "unknown markers removed": count_messages("removed unknown markers"),
    "schema retries": count_messages("did not fit"),
    "transient retries, decision tier": sum("(decision," in m for m in retry_messages),
    "transient retries, writer tier": sum("(writer," in m for m in retry_messages),
    "model requests": f"{call_budget.requests_sent} of {call_budget.limit} allowed",
    "planner seconds": round(planner_seconds, 1),
    "research seconds": round(research_seconds, 1),
    "writer seconds": round(writer_seconds, 1),
    "seconds": round(elapsed_seconds, 1),
}
for measure, value in run_numbers.items():
    print(f"{measure:<34}{value}")
print(f"{'dropped, by reason':<34}{dropped_by_reason() or 'none'}")
print(f"{'skipped, by reason':<34}{skipped_by_reason() or 'none'}")
print(
    f"{'rates (requests a second)':<34}decision {settings.vre_decision_rps}, "
    f"writer {settings.vre_writer_rps}"
)
print("\nBranches finished, seconds after the planner:")
print("  " + ", ".join(f"{qid} {at - planner_seconds:.1f}" for qid, at in branch_finished_at))
print("\nWhere each branch spent its time:")
for message in run_log.messages:
    if ": timing: " in message:
        print(f"  {message}")

In [ ]:
# CONCEPT: tracing-from-day-one. LangSmith uploads runs from a background
# thread; wait until everything is sent, then find this run by its id.
wait_for_all_tracers()

if settings.langsmith_tracing:
    from langsmith import Client

    langsmith_client = Client()
    for lookup_attempt in range(1, 6):
        try:
            traced_run = langsmith_client.read_run(run_id)
            trace_url = langsmith_client.get_run_url(
                run=traced_run, project_name=settings.langsmith_project
            )
            print(f"Trace: {trace_url}")
            break
        except Exception:  # a new run can take a few seconds to appear
            time.sleep(2 * lookup_attempt)
    else:
        print(f"Trace not found yet: search project {settings.langsmith_project} for {run_id}")
else:
    print("Tracing is off (LANGSMITH_TRACING=false)")

In [ ]:
# The committed record of the run. Only the brief and its provenance are
# written: never settings, keys, or LangSmith ids.
prompt_versions = (
    f"P-01 {planner_prompt.VERSION}, P-02 {web_findings_prompt.VERSION}, "
    f"P-03 {writer_prompts.OUTLINE_VERSION}, P-04 {writer_prompts.SECTION_VERSION}"
)
provenance_lines = [
    "> - Generated by `notebooks/phase_03.ipynb`, Phase 3. Same topic as "
    "`phase_01_brief.md` and `phase_02_brief.md`.",
    f"> - Prompts: {prompt_versions}.",
    f"> - Models: `{settings.vre_chat_model}` (planner, extraction), "
    f"`{settings.vre_writer_model}` (writer), at {settings.vre_decision_rps} and "
    f"{settings.vre_writer_rps} requests a second.",
    f"> - Search and page reading: `search-mcp` (MCP), {SEARCH_RESULTS_PER_QUESTION} results "
    f"and up to {PAGES_PER_QUESTION} pages per sub-question, one researcher per sub-question "
    "running in parallel.",
    f"> - Run: {run_numbers['sub-questions']} sub-questions, {run_numbers['findings kept']} "
    f"findings kept, {run_numbers['findings dropped by the checks']} dropped by the checks, "
    f"{run_numbers['sections']} sections, {run_numbers['model requests']} model requests, "
    f"{run_numbers['seconds']} s (research {run_numbers['research seconds']} s, "
    f"writer {run_numbers['writer seconds']} s).",
    "> - Every passage below occurs word for word on the page it cites, checked in code. "
    "Whether each sentence is supported by its passage is checked from phase 5.",
]
title_line, brief_body = brief_markdown.split("\n", 1)
brief_file_text = "\n".join([title_line, "", *provenance_lines, brief_body])

brief_path = PROJECT_ROOT / "notebooks" / "phase_03_brief.md"
brief_path.write_text(brief_file_text, encoding="utf-8")
print(f"Wrote {brief_path.relative_to(PROJECT_ROOT)} ({len(brief_file_text.splitlines())} lines)")

In [ ]:
# Phases 1 and 2 as measured on the runs that closed them; Phase 3 is this run.
comparison_rows = [
    (
        "Where a passage comes from",
        "Search snippet",
        "Checked copy from a page read",
        "Checked copy from a page read",
    ),
    (
        "Sub-questions researched",
        "One after another",
        "One after another",
        f"All at once ({run_numbers['branches run']} branches)",
    ),
    (
        "Model rate limit",
        "One throttle, 1 a second",
        "One throttle, 1 a second",
        f"Per model: {settings.vre_decision_rps} and {settings.vre_writer_rps} a second",
    ),
    ("Findings kept", "18", "14", str(run_numbers["findings kept"])),
    (
        "Sub-questions without evidence",
        "0",
        "0",
        str(run_numbers["sub-questions with no evidence"]),
    ),
    ("Research stage", "", "", f"{run_numbers['research seconds']} s"),
    ("Writer stage", "", "", f"{run_numbers['writer seconds']} s"),
    ("Total time", "43.6 s", "99.4 s", f"{run_numbers['seconds']} s"),
]
table_lines = [
    "| | Phase 1 | Phase 2 | **Phase 3 (this run)** |",
    "| --- | --- | --- | --- |",
    *(
        f"| {measure} | {first} | {second} | **{third}** |"
        for measure, first, second, third in comparison_rows
    ),
]
display(Markdown("\n".join(table_lines)))

## Problems we hit and what we decided

The phase record. One entry per problem met while opening, building, and closing Phase 3: what happened, what caused it (or what a check showed), the options, and the decision. Decision numbers refer to the project's decision log.

### 1. Most of the free tracing quota had gone on tests (D-41)

**What happened.** Reading the free-tier limits before the phase, LangSmith showed 1,584 of the month's 5,000 traces used, with only a few briefs run.
**The check.** The scripted tests were run against a fake LangSmith server that counts what arrives: one local run sent 66 traces of fake model calls. `settings.py` copies `.env`, with tracing on, into the environment, so every local test run uploaded.
**Options.** Tests never trace; only scripted tests stop tracing; a separate project for tests (same quota); tracing off in `.env` and on by hand.
**Decision.** Scripted tests send no traces, live tests keep theirs, and a test fails if that stops being true. Measured again: 0 traces per scripted run. The same decision set the spending caps: no payment card on Mistral or LangSmith, Tavily's pay as you go off, so every free tier stops at its limit instead of charging.

### 2. Keys were visible in a screenshot

**What happened.** A screenshot of the editor, taken for the terminal, also showed the `.env` tab: the Mistral key and most of the Tavily key were readable.
**Decision.** Both keys replaced and the old ones deleted at once, the LangSmith key replaced as planned, and a rule added to the frequent checks: close `.env` before any screenshot, and keep no old keys in it. No card is attached to any account, so the exposure could not cost money.

### 3. An empty list of branches ends a run silently

**What happened.** Before building, a small experiment checked how LangGraph's `Send` behaves. It confirmed the branches run at the same time, and showed that a routing function returning no branches ends the run right there, with no error and no brief.
**Decision.** With no web sub-question the run goes to the join node instead (`parallel-fan-out`, decision 7). It cannot happen yet; from Phase 4 a topic's questions can all be routed to documents.

### 4. A test that passed only sometimes

**What happened.** Once the branches ran in parallel, the Phase 2 end-to-end test failed 4 runs in 15. The fake model gave replies in call order; two branches call it at about the same moment, so one sometimes took the other's reply, and the word-for-word check dropped every passage.
**Decision.** The fake model can key a reply to text in the prompt (the sub-question), so the order of calls no longer matters. Calls made one after another still use call order. **Lesson:** a test double that depends on call order assumes calls happen one at a time.

### 5. A timing test that failed on Windows

**What happened.** A test checked that branches ran at the same time by the total run time: under 0.8 s. It passed on Linux at 0.45 s and failed on Windows at 0.803 s, because the total included scripted model calls whose speed depends on the machine.
**Decision.** The test checks that the searches overlapped (every one started before the first one finished), which holds on any machine. **Lesson:** test concurrency by overlap, not by a time limit.

### 6. Eleven pages skipped: was the fan-out causing timeouts?

**What happened.** The first parallel run skipped 11 of 30 page reads, more than usual, with 30 reads now starting together.
**The check.** A timed-out read takes 15 s; every branch had finished within 13.7 s, search and model call included, so no read could have timed out. The skips were refusals.
**Decision.** No limit on how many branches run at once (open choice A of `parallel-fan-out`). Each branch now logs its timings and the run prints skip reasons by kind, so the next question of this sort is answered directly.

### 7. One throttle could not fit two models (D-43)

**What happened.** One limiter at 1 request a second served both models, while Mistral allows the 8b model 3.13 a second and the 14b model 0.50: the parallel P-02 calls trickled out one a second, and the writer was allowed over its limit.
**Options.** One throttle at the lower or the higher limit; one per model, inside each gateway or shared by the process; the full limits, 80%, or half.
**Decision.** One limiter per model, shared by every gateway in the process, at 80% of each limit (2.5 and 0.4 a second); retries wait as the provider asks, or with a random spread so parallel branches do not retry together; and a budget of 30 model requests per run, carried in the run's config, so a loop cannot keep spending. In two live runs: no rate-limit retry on either model, and 14 requests used.

### 8. A slow run that could not be explained

**What happened.** The first run with the new rates took 58.0 s, its research stage 21.9 s against 13.7 s before. The obvious suspect was page reads reaching the 15 s limit.
**Decision.** Nothing was changed on a guess. Each branch now logs its search, page-read, and model-call times. The next run, measured, took 43.7 s with no timeout: search 1 to 4 s, page reads 4 to 5 s, model call 3 to 5 s per branch. The slow run is recorded as variation from the web and the provider. **Lesson:** measure the parts before changing anything; one run of a system that depends on the web explains little.

### 9. The writer is now the slowest stage

**What happened.** With research down to about 13 s, the writer's seven calls, one after another on the 14b model, take 27 to 33 s, more than half of a brief. The throttle does not slow them: each call takes longer than the 2.5 s gap its limit needs.
**Decision.** Not changed in Phase 3, whose plan is parallel research. Writing sections in parallel is carried as an option for a later phase, to be weighed against the 14b model's low rate limit.

### 10. The closing run took 84.6 s: the provider was out of capacity

**What happened.** This notebook's run took 84.6 s, against 43.7 s for the best run, with one retry on each model. The timings each branch now logs showed searches and page reads at their usual speed and every model call three to five times slower (P-02 15 to 22 s instead of 3 to 5 s).
**The check.** The trace's two failed requests (LangSmith, Tracing, Runs, Status: error) both said `429 ... "Not enough capacity available for this request, please retry later." "type":"backend_out_of_capacity"`. A 429, but Mistral's capacity, not our rate: we send at 80% of the limit, and no lower rate would have prevented it.
**Decision.** No change. The gateway retried both requests after a randomised wait and the run finished, which is NFR-04 ("a single rate-limit response does not fail a run") in a live run. Run time is reported as a range: four runs on this topic took **43.7 to 84.6 s**, all within NFR-02's 10 minutes and all faster than Phase 2's 99.4 s; the spread is the provider's response time. **Lesson:** a 429's `type` says whose limit was reached; read it before changing a rate.